In [ ]:
!pip install pygrog[dev]


# Utils Tour: Coil Compression and NLINV

This example introduces two utility routines from :mod:`pygrog.utils`:

1. **Coil compression** (:func:`~pygrog.utils.coil_compress`) — reduces the
   number of receiver coils via PCA to speed up downstream processing without
   significant SNR loss.
2. **NLINV coil calibration** (:func:`~pygrog.utils.nlinv_calib`) — estimates
   coil sensitivity maps from undersampled k-space data using the
   nonlinear-inverse (NLINV) algorithm.

All data use a **BrainWeb T1-weighted** phantom: a multi-coil dataset is
constructed by multiplying it with synthetic coil sensitivity maps.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from brainweb_dl import get_mri

## BrainWeb multi-coil dataset

We load a BrainWeb T1-weighted phantom and multiply it by synthetic coil
sensitivity maps (smooth Gaussian weighting centred at each coil position).



In [ ]:
def _synthetic_smaps(shape, n_coils=4):
    """Synthetic sensitivity maps (Gaussian envelope + linear phase)."""
    ny, nx = shape
    yy, xx = np.mgrid[-1 : 1 : ny * 1j, -1 : 1 : nx * 1j]
    smaps = []
    for angle in np.linspace(0.0, 2.0 * np.pi, n_coils, endpoint=False):
        cx, cy = np.cos(angle), np.sin(angle)
        gauss = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2.0 * 0.45**2))
        phase = np.exp(1j * (cx * xx + cy * yy))
        smaps.append(gauss * phase)
    smaps = np.asarray(smaps, dtype=np.complex64)
    smaps /= np.sqrt((np.abs(smaps) ** 2).sum(0, keepdims=True)) + 1e-12
    return smaps


image = get_mri(0, "T1")
image = np.flip(image, axis=(0, 2))[90].astype(np.float32)
image /= image.max() + 1e-8
image_shape = image.shape
n_coils = 16
smaps_gt = _synthetic_smaps(image_shape, n_coils=n_coils)

# Coil images: (n_coils, ny, nx)
coil_images = smaps_gt * image[np.newaxis].astype(np.complex64)

# Cartesian k-space: (n_coils, ny, nx)
kspace_full = np.fft.fftshift(
    np.fft.fft2(np.fft.ifftshift(coil_images, axes=(-2, -1))), axes=(-2, -1)
)

print(f"Coil images shape: {coil_images.shape}")
print(f"K-space shape    : {kspace_full.shape}")

In [ ]:
fig, axes = plt.subplots(2, n_coils // 2, figsize=(11, 4))
axes = axes.flatten()
for i in range(n_coils):
    axes[i].imshow(np.abs(coil_images[i]), cmap="gray", origin="lower")
    axes[i].set_title(f"Coil {i + 1}")
    axes[i].axis("off")
plt.suptitle("Synthetic multi-coil images (16 coils)")
plt.tight_layout()
plt.show()

## Coil compression

:func:`~pygrog.utils.coil_compress` reduces the number of virtual coils
using PCA on the k-space data.  It returns the compressed k-space and the
compression matrix ``W`` of shape ``(n_virtual, n_coils)``.

The argument ``n_coils`` accepts:

* an **integer** — exact number of virtual coils to keep, or
* a **float** in (0, 1] — energy fraction to retain.



In [ ]:
from pygrog.utils import coil_compress

n_virtual = 8

kspace_flat = kspace_full.reshape(n_coils, -1)  # (n_coils, n_samples)

kspace_cc, W = coil_compress(kspace_flat, n_virtual)

print(f"\nOriginal coils   : {kspace_flat.shape[0]}")
print(f"Virtual coils    : {kspace_cc.shape[0]}")
print(f"Compression matrix: {W.shape}")

# Reconstruct coil images from compressed data
kspace_cc_full = kspace_cc.reshape(n_virtual, *image_shape)
coil_images_cc = np.fft.fftshift(
    np.fft.ifft2(np.fft.ifftshift(kspace_cc_full, axes=(-2, -1))), axes=(-2, -1)
)

# RSS of compressed coils vs original
rss_orig = np.sqrt((np.abs(coil_images) ** 2).sum(0))
rss_cc = np.sqrt((np.abs(coil_images_cc) ** 2).sum(0))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(image, cmap="gray", origin="lower")
axes[0].set_title("Reference")
axes[0].axis("off")
axes[1].imshow(rss_orig, cmap="gray", origin="lower")
axes[1].set_title(f"RSS ({n_coils} coils)")
axes[1].axis("off")
axes[2].imshow(rss_cc, cmap="gray", origin="lower")
axes[2].set_title(f"RSS ({n_virtual} virtual coils, PCA)")
axes[2].axis("off")
plt.suptitle("Coil compression")
plt.tight_layout()
plt.show()

<div class="alert alert-info"><h4>Note</h4><p>Coil compression is lossless when ``n_coils >= original_n_coils`` and
   near-lossless when the retained energy fraction is high (e.g. > 0.99).
   Use a float threshold ``coil_compress(data, 0.99)`` for automatic rank
   selection.</p></div>



## NLINV coil sensitivity estimation

:func:`~pygrog.utils.nlinv_calib` estimates coil sensitivity maps from a
Cartesian **undersampled** acquisition using the nonlinear-inverse (NLINV)
algorithm (Uecker et al.\ 2008).  It alternates between estimating the image
and the sensitivities, coupled by a Sobolev-space regulariser on the maps.
Crucially, **no separate autocalibration (ACS) region is required** — NLINV
estimates both the image and the maps jointly from the undersampled data.

The function returns ``(smaps, image)`` when ``ret_image=True``, or just
``smaps`` otherwise.



In [ ]:
from pygrog.utils import nlinv_calib

# Cartesian undersampling mask: regular acceleration x2 on phase-encode
mask = np.zeros(image_shape, dtype=bool)
mask[::2, :] = True  # keep every other phase-encode line

# Apply mask to k-space
kspace_us = kspace_full * mask[np.newaxis]

print(f"\nUndersampling factor: {mask.size / mask.sum():.1f}x")
print(f"Undersampled k-space shape: {kspace_us.shape}")

In [ ]:
# NLINV calibration (Cartesian, pass mask and ndim)
smaps_nlinv, _, image_nlinv = nlinv_calib(
    kspace_us,
    ndim=2,
    mask=mask,
    max_iter=8,
    cg_iter=5,
    ret_cal=True,
    ret_image=True,
)

print(f"\nEstimated smaps shape: {smaps_nlinv.shape}")
print(f"Reconstructed image shape: {image_nlinv.shape}")

In [ ]:
fig, axes = plt.subplots(2, n_coils // 2 + 1, figsize=(11, 5))
axes = axes.flatten()

axes[0].imshow(image, cmap="gray", origin="lower")
axes[0].set_title("Reference")
axes[0].axis("off")

for i in range(n_coils // 2):
    axes[i + 1].imshow(np.abs(smaps_gt[i]), cmap="magma", origin="lower", vmin=0)
    axes[i + 1].set_title(f"GT smap {i + 1}")
    axes[i + 1].axis("off")

smaps_nlinv_np = (
    smaps_nlinv.numpy() if isinstance(smaps_nlinv, torch.Tensor) else smaps_nlinv
)
axes[n_coils // 2 + 1].imshow(np.abs(image_nlinv), cmap="gray", origin="lower")
axes[n_coils // 2 + 1].set_title("NLINV image")
axes[n_coils // 2 + 1].axis("off")

for i in range(n_coils // 2):
    axes[n_coils // 2 + 2 + i].imshow(
        np.abs(smaps_nlinv_np[i]), cmap="magma", origin="lower", vmin=0
    )
    axes[n_coils // 2 + 2 + i].set_title(f"NLINV smap {i + 1}")
    axes[n_coils // 2 + 2 + i].axis("off")

plt.suptitle("NLINV coil sensitivity estimation")
plt.tight_layout()
plt.show()

<div class="alert alert-info"><h4>Note</h4><p>NLINV is an iterative method; accuracy improves with more ``max_iter``
   and ``cg_iter`` at the cost of compute time.  The Sobolev-space
   regulariser (``sobolev_width``, ``sobolev_deg``) enforces smooth
   sensitivity maps — increase ``sobolev_deg`` for smoother maps at the
   cost of accuracy near the FOV boundary.</p></div>



## Synthetic calibration k-space from NLINV

By default (``ret_cal=True``) :func:`~pygrog.utils.nlinv_calib` also returns
a **synthesized calibration k-space** (``grappa_train``) — a fully-sampled
Cartesian calibration region reconstructed from the NLINV image and
sensitivity estimates.  This is exactly the input expected by GRAPPA/GROG
calibration kernels, making it easy to plug NLINV into non-Cartesian
pipelines where no explicit calibration scan is available.



In [ ]:
smaps_nlinv2, grappa_train, image_nlinv2 = nlinv_calib(
    kspace_us,
    ndim=2,
    mask=mask,
    max_iter=8,
    cg_iter=5,
    ret_cal=True,  # default — return synthetic calibration k-space
    ret_image=True,
)

grappa_train_np = (
    grappa_train.numpy() if isinstance(grappa_train, torch.Tensor) else grappa_train
)

print(f"\nSynthetic calibration k-space shape: {grappa_train_np.shape}")
print(f"  coils x cal_height x cal_width : {grappa_train_np.shape}")

In [ ]:
cal_height, cal_width = grappa_train_np.shape[-2], grappa_train_np.shape[-1]
ncols = min(n_coils // 2, 4)

fig, axes = plt.subplots(2, ncols, figsize=(3 * ncols, 5.5))
for i in range(ncols):
    # top row: log-magnitude of calibration k-space
    axes[0, i].imshow(
        np.log1p(np.abs(grappa_train_np[i])), cmap="inferno", origin="lower"
    )
    axes[0, i].set_title(f"Calib k-space (log) — coil {i + 1}")
    axes[0, i].axis("off")
    # bottom row: IFFT of calibration region (coil PSF)
    cal_img = np.fft.fftshift(
        np.fft.ifft2(np.fft.ifftshift(grappa_train_np[i], axes=(-2, -1))),
        axes=(-2, -1),
    )
    axes[1, i].imshow(np.abs(cal_img), cmap="gray", origin="lower")
    axes[1, i].set_title(f"Cal image — coil {i + 1}")
    axes[1, i].axis("off")

plt.suptitle(
    f"NLINV synthetic calibration k-space  ({cal_height}x{cal_width} centre region)"
)
plt.tight_layout()
plt.show()

<div class="alert alert-info"><h4>Note</h4><p>The synthetic calibration region is computed from the NLINV image and
   sensitivity estimates as ``FFT(image x smap)`` for each coil, cropped to
   the central ``cal_width x cal_width`` lines.  Its quality therefore
   depends on the NLINV reconstruction quality — more iterations give a more
   faithful calibration dataset.</p></div>



In [ ]:
plt.show()